<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/07_holdout_review_policy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Reproduce proposed holdout and clean training view.
# Read-only: no CSVs or reference labels are modified.

import hashlib
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

def load_text_data(relative_path):
    # Read ONLY identifiers and article text.
    return pd.read_csv(
        BASE / relative_path,
        usecols=["record_id", "text"]
    )

noisy = load_text_data(
    "data/splits/noisy_training.csv"
)
pilot = load_text_data(
    "data/pilot/pilot_500.csv"
)
anchor = load_text_data(
    "data/splits/anchor.csv"
)
development = load_text_data(
    "data/splits/development.csv"
)

for name, df in {
    "noisy": noisy,
    "pilot": pilot,
    "anchor": anchor,
    "development": development
}.items():
    assert len(df) == df["record_id"].nunique(), name
    assert df["text"].notna().all(), name

noisy_ids = set(noisy["record_id"].astype(int))
pilot_ids = set(pilot["record_id"].astype(int))

assert len(noisy_ids) == 16200
assert len(pilot_ids) == 500
assert pilot_ids.issubset(noisy_ids)

# Exact same selection procedure as Day 6.
SEED = 20261025
available_ids = np.array(
    sorted(noisy_ids - pilot_ids),
    dtype=np.int64
)

rng = np.random.default_rng(SEED)

holdout_ids = set(
    rng.choice(
        available_ids,
        size=2000,
        replace=False
    ).tolist()
)

training_ids = noisy_ids - pilot_ids - holdout_ids

holdout_df = noisy[
    noisy["record_id"].isin(holdout_ids)
].copy()

training_df = noisy[
    noisy["record_id"].isin(training_ids)
].copy()

def fingerprint_ids(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

assert fingerprint_ids(holdout_ids) == (
    "37bd15b5fb600e7cfc392c936fcb"
    "187b0a39313c3502098c0e6a403153057051"
)

def normalized_hashes(df):
    normalized = (
        df["text"].astype(str)
        .str.normalize("NFKC")
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    assert normalized.ne("").all()

    return normalized.map(
        lambda s: hashlib.sha256(
            s.encode("utf-8")
        ).hexdigest()
    )

# Protect anchor, development, pilot and holdout.
protected_hashes = set()

for df in [anchor, development, pilot, holdout_df]:
    protected_hashes.update(normalized_hashes(df))

candidate = training_df.sort_values(
    "record_id"
).copy()

candidate["_hash"] = normalized_hashes(
    candidate
).to_numpy()

# Eliminate protected overlaps, then retain
# the lowest record_id per training text.
clean_training_df = (
    candidate[
        ~candidate["_hash"].isin(protected_hashes)
    ]
    .drop_duplicates("_hash", keep="first")
    .drop(columns="_hash")
    .copy()
)

removed_ids = (
    training_ids -
    set(clean_training_df["record_id"].astype(int))
)

assert removed_ids == {
    9266, 43231, 68708, 79605
}

clean_ids = set(
    clean_training_df["record_id"].astype(int)
)

assert fingerprint_ids(clean_ids) == (
    "52176ba0a8d8d43415ff34c083"
    "f75de26f9db05b73458893a23413998aa28b88"
)

assert len(holdout_df) == 2000
assert len(clean_training_df) == 13696
assert holdout_ids.isdisjoint(clean_ids)

print("DAY 7 REPRODUCIBILITY CHECK PASSED")
print("----------------------------------")
print("Clean anchor:", len(anchor))
print("Development:", len(development))
print("Exploratory pilot:", len(pilot))
print("Proposed holdout:", len(holdout_df))
print("Clean training candidate:", len(clean_training_df))
print("Duplicate exclusions:", sorted(removed_ids))
print("Day 6 selection fingerprints: MATCHED")
print("Private noisy reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


Mounted at /content/drive
DAY 7 REPRODUCIBILITY CHECK PASSED
----------------------------------
Clean anchor: 2000
Development: 1800
Exploratory pilot: 500
Proposed holdout: 2000
Clean training candidate: 13696
Duplicate exclusions: [9266, 43231, 68708, 79605]
Day 6 selection fingerprints: MATCHED
Private noisy reference labels accessed: NO
Files modified: NO
Holdout frozen: NO


In [3]:

# Rebuild the near-duplicate review queue.
# Text-only screening; no reference labels or file writes.

import numpy as np
import pandas as pd
import re
import unicodedata

from difflib import SequenceMatcher
from sklearn.feature_extraction.text import TfidfVectorizer

# Reconstruct yesterday's comparison order
holdout_check = holdout_df[
    ["record_id", "text"]
].reset_index(drop=True)

comparison = pd.concat(
    [
        anchor.assign(partition="anchor"),
        development.assign(partition="development"),
        clean_training_df.assign(partition="training"),
        pilot.assign(partition="pilot")
    ],
    ignore_index=True
)[["record_id", "text", "partition"]]

assert len(holdout_check) == 2000
assert len(comparison) == 17996
assert comparison["record_id"].is_unique

# Same TF-IDF configuration as Day 6
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=80000,
    sublinear_tf=True,
    dtype=np.float32
)

all_texts = pd.concat(
    [holdout_check["text"], comparison["text"]],
    ignore_index=True
).astype(str)

X = vectorizer.fit_transform(all_texts)

X_holdout = X[:2000]
X_comparison = X[2000:]

TOP_K = 5
THRESHOLD = 0.85
BATCH = 100

rows = []

for start in range(0, 2000, BATCH):
    end = min(start + BATCH, 2000)

    # L2-normalized TF-IDF: dot product = cosine
    similarity = (
        X_holdout[start:end] @ X_comparison.T
    ).toarray()

    top_indices = np.argpartition(
        similarity, -TOP_K, axis=1
    )[:, -TOP_K:]

    for local_i in range(end - start):
        indices = sorted(
            top_indices[local_i],
            key=lambda j: (
                -float(similarity[local_i, j]),
                int(comparison.iloc[j]["record_id"])
            )
        )

        for rank, j in enumerate(indices, start=1):
            score = float(similarity[local_i, j])

            if score < THRESHOLD:
                continue

            match = comparison.iloc[j]

            rows.append({
                "holdout_id": int(
                    holdout_check.iloc[start + local_i]["record_id"]
                ),
                "matching_id": int(match["record_id"]),
                "partition": match["partition"],
                "rank": rank,
                "cosine": score
            })

review_queue = pd.DataFrame(rows)

assert not review_queue.empty
assert not review_queue.duplicated(
    ["holdout_id", "matching_id"]
).any()

# Text comparison using IDs, without labels
h_texts = holdout_check.set_index("record_id")["text"]
c_texts = comparison.set_index("record_id")["text"]

def normalize_text(text):
    return re.sub(
        r"\s+", " ",
        unicodedata.normalize(
            "NFKC", str(text)
        ).casefold()
    ).strip()

def compare_pair(row):
    a = normalize_text(h_texts.loc[row.holdout_id])
    b = normalize_text(c_texts.loc[row.matching_id])

    sequence = SequenceMatcher(
        None, a, b, autojunk=False
    ).ratio()

    wa, wb = set(a.split()), set(b.split())
    union = wa | wb
    jaccard = len(wa & wb) / len(union) if union else 1.0

    return pd.Series({
        "sequence": sequence,
        "jaccard": jaccard,
        "exact": a == b
    })

review_queue = pd.concat(
    [
        review_queue.reset_index(drop=True),
        review_queue.apply(compare_pair, axis=1)
    ],
    axis=1
)

# Priority is a review-order heuristic ONLY
protected = {"anchor", "development", "pilot"}

review_queue["priority"] = review_queue.apply(
    lambda r: (
        "P1_protected"
        if r["partition"] in protected
        else "P2_training"
    ),
    axis=1
)

review_queue["strong_screen"] = (
    (review_queue["sequence"] >= 0.90)
    & (review_queue["jaccard"] >= 0.85)
)

review_queue["review_status"] = "PENDING"

review_queue = review_queue.sort_values(
    ["priority", "cosine", "holdout_id"],
    ascending=[True, False, True]
).reset_index(drop=True)

unique_holdouts = review_queue["holdout_id"].nunique()


print("Candidate pairs:", len(review_queue))
print("Unique holdout IDs:", unique_holdouts)
print("Protected-partition pairs:", int(
    review_queue["partition"].isin(protected).sum()
))
print("Training pairs:", int(
    (review_queue["partition"] == "training").sum()
))
print("Strong screening pairs:", int(
    review_queue["strong_screen"].sum()
))
print("Confirmed duplicates: NOT YET ASSESSED")

if len(review_queue) != 63 or unique_holdouts != 57:
    print("WARNING: Counts differ from Day 6; investigate.")

print("\nREVIEW QUEUE — TOP 15")
display(
    review_queue.head(15).round(5)
)

print("\nReview status: ALL PENDING")
print("Reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


Candidate pairs: 63
Unique holdout IDs: 57
Protected-partition pairs: 18
Training pairs: 45
Strong screening pairs: 15
Confirmed duplicates: NOT YET ASSESSED

REVIEW QUEUE — TOP 15


,holdout_id,matching_id,partition,rank,cosine,sequence,jaccard,exact,priority,strong_screen,review_status
0,117637,117330,development,1,0.99258,0.98069,0.92105,False,P1_protected,True,PENDING
1,98008,97990,anchor,1,0.98932,0.95175,0.88235,False,P1_protected,True,PENDING
2,107847,107824,anchor,1,0.98194,0.95522,0.94444,False,P1_protected,True,PENDING
3,90178,90148,development,1,0.97965,0.95701,0.75000,False,P1_protected,False,PENDING
4,8972,9101,anchor,1,0.97701,0.96367,0.92500,False,P1_protected,True,PENDING
5,51212,51256,pilot,1,0.97473,0.96071,0.76744,False,P1_protected,False,PENDING
6,18762,18760,anchor,1,0.94556,0.94737,0.72414,False,P1_protected,False,PENDING
7,33781,33762,anchor,1,0.91553,0.86471,0.83019,False,P1_protected,False,PENDING
8,6540,4514,development,1,0.91100,0.89573,0.54902,False,P1_protected,False,PENDING
9,112808,105416,development,1,0.90638,0.85256,0.54639,False,P1_protected,False,PENDING



Review status: ALL PENDING
Reference labels accessed: NO
Files modified: NO
Holdout frozen: NO


In [4]:

# Intra-Holdout Near-Duplicate Detection
# Text-only audit. No labels or file modifications.

import numpy as np
import pandas as pd
from difflib import SequenceMatcher

assert len(holdout_check) == 2000
assert X_holdout.shape[0] == 2000

TOP_K = 5
THRESHOLD = 0.85
BATCH_SIZE = 100

internal_pairs = {}

for start in range(0, len(holdout_check), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(holdout_check))

    similarities = (
        X_holdout[start:end] @ X_holdout.T
    ).toarray()

    # Exclude each article's comparison with itself.
    similarities[
        np.arange(end - start),
        np.arange(start, end)
    ] = -1.0

    nearest_indices = np.argpartition(
        similarities,
        -TOP_K,
        axis=1
    )[:, -TOP_K:]

    for local_i in range(end - start):
        source_idx = start + local_i

        source_id = int(
            holdout_check.iloc[source_idx]["record_id"]
        )

        for match_idx in nearest_indices[local_i]:
            score = float(
                similarities[local_i, match_idx]
            )

            if score < THRESHOLD:
                continue

            match_id = int(
                holdout_check.iloc[match_idx]["record_id"]
            )

            # Canonical pair order avoids counting
            # A-B and B-A as separate pairs.
            pair = tuple(sorted((source_id, match_id)))

            internal_pairs[pair] = max(
                internal_pairs.get(pair, -1.0),
                score
            )

holdout_internal_review = pd.DataFrame(
    [
        {
            "record_a": pair[0],
            "record_b": pair[1],
            "cosine": score
        }
        for pair, score in internal_pairs.items()
    ],
    columns=["record_a", "record_b", "cosine"]
)

# Compute additional text similarity scores
# for every candidate pair.
holdout_text_lookup = holdout_check.set_index(
    "record_id"
)["text"]

def assess_internal_pair(row):
    a = normalize_text(
        holdout_text_lookup.loc[row.record_a]
    )

    b = normalize_text(
        holdout_text_lookup.loc[row.record_b]
    )

    sequence = SequenceMatcher(
        None, a, b, autojunk=False
    ).ratio()

    tokens_a = set(a.split())
    tokens_b = set(b.split())
    union = tokens_a | tokens_b

    jaccard = (
        len(tokens_a & tokens_b) / len(union)
        if union else 1.0
    )

    return pd.Series({
        "sequence": sequence,
        "jaccard": jaccard,
        "normalized_exact": a == b
    })

if not holdout_internal_review.empty:
    metrics = holdout_internal_review.apply(
        assess_internal_pair,
        axis=1
    )

    holdout_internal_review = pd.concat(
        [
            holdout_internal_review.reset_index(drop=True),
            metrics.reset_index(drop=True)
        ],
        axis=1
    )

    holdout_internal_review = (
        holdout_internal_review
        .sort_values(
            ["cosine", "record_a", "record_b"],
            ascending=[False, True, True]
        )
        .reset_index(drop=True)
    )

    internal_ids = set(
        holdout_internal_review["record_a"]
    ) | set(
        holdout_internal_review["record_b"]
    )

else:
    internal_ids = set()

# Cross-check exact duplicates independently
# of TF-IDF top-k screening.
internal_hashes = normalized_hashes(holdout_check)

exact_duplicate_count = int(
    internal_hashes.duplicated().sum()
)

assert exact_duplicate_count == 0

print("INTRA-HOLDOUT DUPLICATE AUDIT COMPLETED")
print("---------------------------------------")
print("Holdout records:", len(holdout_check))
print("Top neighbors checked:", TOP_K)
print(
    "Unique candidate pairs >= 0.85:",
    len(holdout_internal_review)
)
print(
    "Unique involved records:",
    len(internal_ids)
)
print(
    "Normalized exact duplicate rows:",
    exact_duplicate_count
)

print("\nTOP 15 INTERNAL CANDIDATES")
display(
    holdout_internal_review.head(15).round(5)
)

print("\nExternal review queue:", len(review_queue))
print("Reference labels accessed: NO")
print("Original files modified: NO")
print("Holdout frozen: NO")
print("Human review: PENDING")


INTRA-HOLDOUT DUPLICATE AUDIT COMPLETED
---------------------------------------
Holdout records: 2000
Top neighbors checked: 5
Unique candidate pairs >= 0.85: 8
Unique involved records: 8
Normalized exact duplicate rows: 0

TOP 15 INTERNAL CANDIDATES


,record_a,record_b,cosine,sequence,jaccard,normalized_exact
0,92770,92798,0.90385,0.86034,0.67857,False
1,56820,56966,0.89467,0.87666,0.72093,False
2,95309,112808,0.89205,0.84287,0.58427,False
3,111618,112808,0.88620,0.84243,0.55789,False
4,106486,112808,0.87778,0.84364,0.52941,False
5,95309,111618,0.87665,0.87290,0.62069,False
6,95309,106486,0.86799,0.84470,0.56842,False
7,106486,111618,0.86112,0.83409,0.51456,False



External review queue: 63
Reference labels accessed: NO
Original files modified: NO
Holdout frozen: NO
Human review: PENDING


In [5]:

# Near-Duplicate Cluster Identification
# Graph-based review organization, without labels.

import pandas as pd
from collections import defaultdict

assert len(review_queue) == 63
assert len(holdout_internal_review) == 8

parent = {}

def find(node):
    if node not in parent:
        parent[node] = node

    if parent[node] != node:
        parent[node] = find(parent[node])

    return parent[node]

def union(a, b):
    ra, rb = find(a), find(b)

    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

# External near-duplicate candidate edges
for row in review_queue.itertuples(index=False):
    union(
        ("holdout", int(row.holdout_id)),
        (str(row.partition), int(row.matching_id))
    )

# Internal holdout candidate edges
for row in holdout_internal_review.itertuples(index=False):
    union(
        ("holdout", int(row.record_a)),
        ("holdout", int(row.record_b))
    )

components = defaultdict(list)

for node in list(parent):
    components[find(node)].append(node)

summary_rows = []

for nodes in components.values():
    nodes = sorted(nodes)
    holdout_nodes = [
        record_id for partition, record_id in nodes
        if partition == "holdout"
    ]
    protected_nodes = [
        (partition, record_id)
        for partition, record_id in nodes
        if partition in {"anchor", "development", "pilot"}
    ]

    partitions = sorted(set(
        partition for partition, _ in nodes
    ))

    summary_rows.append({
        "total_nodes": len(nodes),
        "holdout_count": len(holdout_nodes),
        "protected_count": len(protected_nodes),
        "contains_protected": bool(protected_nodes),
        "partitions": ", ".join(partitions),
        "holdout_ids": ", ".join(
            str(x) for x in sorted(holdout_nodes)
        )
    })

cluster_summary = pd.DataFrame(summary_rows)

cluster_summary = cluster_summary.sort_values(
    ["contains_protected", "total_nodes", "holdout_count"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Verify the four-record internal cluster
special_ids = [95309, 112808, 106486, 111618]

assert len({
    find(("holdout", x))
    for x in special_ids
}) == 1

# All previously flagged holdout IDs must appear
flagged_holdouts = (
    set(review_queue["holdout_id"].astype(int))
    | set(holdout_internal_review["record_a"].astype(int))
    | set(holdout_internal_review["record_b"].astype(int))
)

covered_holdouts = {
    record_id
    for nodes in components.values()
    for partition, record_id in nodes
    if partition == "holdout"
}

assert flagged_holdouts == covered_holdouts

print("NEAR-DUPLICATE CLUSTER ANALYSIS COMPLETED")
print("-----------------------------------------")
print("External candidate edges:", len(review_queue))
print("Internal candidate edges:", len(holdout_internal_review))
print("Total candidate edges:", 71)
print("Connected review components:", len(components))
print("Unique flagged holdout records:", len(covered_holdouts))

print(
    "Components containing protected records:",
    int(cluster_summary["contains_protected"].sum())
)

print("\nTOP 15 REVIEW COMPONENTS")
display(cluster_summary.head(15))

print("\nFour-record internal cluster verified: YES")
print("Human adjudication: PENDING")
print("Reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


NEAR-DUPLICATE CLUSTER ANALYSIS COMPLETED
-----------------------------------------
External candidate edges: 63
Internal candidate edges: 8
Total candidate edges: 71
Connected review components: 56
Unique flagged holdout records: 61
Components containing protected records: 15

TOP 15 REVIEW COMPONENTS


,total_nodes,holdout_count,protected_count,contains_protected,partitions,holdout_ids
0,6,4,1,True,"development, holdout, training","95309, 106486, 111618, 112808"
1,3,1,1,True,"anchor, holdout, training",8972
2,3,1,1,True,"anchor, holdout, training",84216
3,2,1,1,True,"development, holdout",117637
4,2,1,1,True,"anchor, holdout",98008
5,2,1,1,True,"anchor, holdout",107847
6,2,1,1,True,"development, holdout",90178
7,2,1,1,True,"holdout, pilot",51212
8,2,1,1,True,"anchor, holdout",18762
9,2,1,1,True,"anchor, holdout",33781



Four-record internal cluster verified: YES
Human adjudication: PENDING
Reference labels accessed: NO
Files modified: NO
Holdout frozen: NO


In [6]:

# Review Priority and Adjudication Queue
# Metadata only; no labels, text export or file writes.

import pandas as pd

PROTECTED_CALIBRATION = {"anchor", "development"}

component_rows = []

# Stable ordering independent of dictionary insertion order
ordered_components = sorted(
    [sorted(nodes) for nodes in components.values()],
    key=lambda nodes: tuple(nodes)
)

node_to_component = {}

for idx, nodes in enumerate(ordered_components, start=1):

    component_id = f"C{idx:03d}"

    for node in nodes:
        assert node not in node_to_component
        node_to_component[node] = component_id

    partitions = {
        partition for partition, _ in nodes
    }

    holdout_ids_in_component = sorted(
        record_id
        for partition, record_id in nodes
        if partition == "holdout"
    )

    if partitions & PROTECTED_CALIBRATION:
        priority = "P1_CALIBRATION"
    elif "pilot" in partitions:
        priority = "P2_PILOT"
    else:
        priority = "P3_TRAINING_INTERNAL"

    component_rows.append({
        "component_id": component_id,
        "priority": priority,
        "total_records": len(nodes),
        "holdout_records": len(holdout_ids_in_component),
        "partitions": ", ".join(sorted(partitions)),
        "holdout_ids": ", ".join(
            map(str, holdout_ids_in_component)
        ),
        "review_status": "PENDING",
        "review_decision": "UNASSESSED"
    })

review_components = pd.DataFrame(component_rows)

# Match external and internal candidate edges
external_edges = [
    (
        ("holdout", int(row.holdout_id)),
        (str(row.partition), int(row.matching_id))
    )
    for row in review_queue.itertuples(index=False)
]

internal_edges = [
    (
        ("holdout", int(row.record_a)),
        ("holdout", int(row.record_b))
    )
    for row in holdout_internal_review.itertuples(index=False)
]

all_edges = external_edges + internal_edges

assert len(all_edges) == 71

# Every edge must be inside one connected component
for node_a, node_b in all_edges:
    assert node_to_component[node_a] == (
        node_to_component[node_b]
    )

# Exact observed results from previous screening
assert len(review_components) == 56

unique_holdout_ids = {
    record_id
    for partition, record_id in node_to_component
    if partition == "holdout"
}

assert len(unique_holdout_ids) == 61

protected_components = sum(
    any(
        partition in {"anchor", "development", "pilot"}
        for partition, _ in nodes
    )
    for nodes in ordered_components
)

assert protected_components == 15

# Stable review ordering
priority_order = {
    "P1_CALIBRATION": 0,
    "P2_PILOT": 1,
    "P3_TRAINING_INTERNAL": 2
}

review_components["_order"] = (
    review_components["priority"].map(priority_order)
)

review_components = (
    review_components.sort_values(
        ["_order", "total_records", "component_id"],
        ascending=[True, False, True]
    )
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("REVIEW PRIORITIZATION VALIDATED")
print("--------------------------------")
print("Total review components:", len(review_components))
print("Unique flagged holdout IDs:", len(unique_holdout_ids))
print("Components with protected records:", protected_components)

print("\nPRIORITY DISTRIBUTION")
display(
    review_components["priority"]
    .value_counts()
    .rename_axis("priority")
    .reset_index(name="components")
)

print("\nTOP 15 REVIEW COMPONENTS")
display(review_components.head(15))

print("\nHuman-reviewed components: 0")
print("Automatic removal performed: NO")
print("Reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


REVIEW PRIORITIZATION VALIDATED
--------------------------------
Total review components: 56
Unique flagged holdout IDs: 61
Components with protected records: 15

PRIORITY DISTRIBUTION


,priority,components
0,P3_TRAINING_INTERNAL,41
1,P1_CALIBRATION,13
2,P2_PILOT,2



TOP 15 REVIEW COMPONENTS


,component_id,priority,total_records,holdout_records,partitions,holdout_ids,review_status,review_decision
0,C011,P1_CALIBRATION,6,4,"development, holdout, training","95309, 106486, 111618, 112808",PENDING,UNASSESSED
1,C001,P1_CALIBRATION,3,1,"anchor, holdout, training",8972,PENDING,UNASSESSED
2,C004,P1_CALIBRATION,3,1,"anchor, holdout, training",84216,PENDING,UNASSESSED
3,C002,P1_CALIBRATION,2,1,"anchor, holdout",18762,PENDING,UNASSESSED
4,C003,P1_CALIBRATION,2,1,"anchor, holdout",33781,PENDING,UNASSESSED
5,C005,P1_CALIBRATION,2,1,"anchor, holdout",98008,PENDING,UNASSESSED
6,C006,P1_CALIBRATION,2,1,"anchor, holdout",103583,PENDING,UNASSESSED
7,C007,P1_CALIBRATION,2,1,"anchor, holdout",107847,PENDING,UNASSESSED
8,C008,P1_CALIBRATION,2,1,"development, holdout",6540,PENDING,UNASSESSED
9,C009,P1_CALIBRATION,2,1,"development, holdout",34202,PENDING,UNASSESSED



Human-reviewed components: 0
Automatic removal performed: NO
Reference labels accessed: NO
Files modified: NO
Holdout frozen: NO


In [7]:

# Pair-Level Human Adjudication Protocol
# Prepare a reproducible review table.
# No automatic decisions, labels or file writes.

import pandas as pd

VALID_DECISIONS = {
    "DUPLICATE",
    "RELATED_DISTINCT",
    "DIFFERENT",
    "UNCERTAIN"
}

assert len(review_queue) == 63
assert len(holdout_internal_review) == 8

adjudication_rows = []

# External candidate pairs
for row in review_queue.itertuples(index=False):

    node_a = ("holdout", int(row.holdout_id))
    node_b = (str(row.partition), int(row.matching_id))

    assert node_to_component[node_a] == (
        node_to_component[node_b]
    )

    adjudication_rows.append({
        "component_id": node_to_component[node_a],
        "pair_type": "EXTERNAL",
        "record_a": int(row.holdout_id),
        "record_b": int(row.matching_id),
        "partition_b": str(row.partition),
        "cosine": float(row.cosine),
        "sequence": float(row.sequence),
        "jaccard": float(row.jaccard)
    })

# Internal holdout candidate pairs
for row in holdout_internal_review.itertuples(index=False):

    node_a = ("holdout", int(row.record_a))
    node_b = ("holdout", int(row.record_b))

    assert node_to_component[node_a] == (
        node_to_component[node_b]
    )

    adjudication_rows.append({
        "component_id": node_to_component[node_a],
        "pair_type": "INTERNAL",
        "record_a": int(row.record_a),
        "record_b": int(row.record_b),
        "partition_b": "holdout",
        "cosine": float(row.cosine),
        "sequence": float(row.sequence),
        "jaccard": float(row.jaccard)
    })

adjudication_df = pd.DataFrame(adjudication_rows)

assert len(adjudication_df) == 71

# Attach component priority
priority_lookup = review_components.set_index(
    "component_id"
)["priority"]

adjudication_df["priority"] = (
    adjudication_df["component_id"].map(
        priority_lookup
    )
)

assert adjudication_df["priority"].notna().all()

# Create stable review identifiers
priority_rank = {
    "P1_CALIBRATION": 0,
    "P2_PILOT": 1,
    "P3_TRAINING_INTERNAL": 2
}

adjudication_df["_priority_rank"] = (
    adjudication_df["priority"].map(priority_rank)
)

adjudication_df = (
    adjudication_df.sort_values(
        [
            "_priority_rank",
            "component_id",
            "cosine",
            "record_a",
            "record_b"
        ],
        ascending=[True, True, False, True, True]
    )
    .drop(columns="_priority_rank")
    .reset_index(drop=True)
)

adjudication_df.insert(
    0,
    "pair_id",
    [
        f"PAIR-{i:03d}"
        for i in range(1, len(adjudication_df) + 1)
    ]
)

# Decisions deliberately remain unassigned
adjudication_df["reviewer_1"] = "PENDING"
adjudication_df["reviewer_2"] = "PENDING"
adjudication_df["final_decision"] = "UNASSESSED"
adjudication_df["review_notes"] = ""

# Integrity checks
assert adjudication_df["pair_id"].is_unique

assert len(
    adjudication_df[
        adjudication_df["pair_type"] == "EXTERNAL"
    ]
) == 63

assert len(
    adjudication_df[
        adjudication_df["pair_type"] == "INTERNAL"
    ]
) == 8

assert adjudication_df[
    "final_decision"
].eq("UNASSESSED").all()

print("ADJUDICATION PROTOCOL VALIDATED")
print("--------------------------------")
print("Total review pairs:", len(adjudication_df))
print("External pairs:", 63)
print("Internal pairs:", 8)
print("Unique components:", adjudication_df[
    "component_id"
].nunique())

print("\nPAIR DISTRIBUTION BY PRIORITY")
display(
    adjudication_df.groupby(
        ["priority", "pair_type"]
    ).size().reset_index(name="pairs")
)

print("\nFIRST 12 ADJUDICATION RECORDS")
display(
    adjudication_df.head(12).round(5)
)

print("\nHuman decisions recorded: 0")
print("Automatic duplicate removal: NO")
print("Reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


ADJUDICATION PROTOCOL VALIDATED
--------------------------------
Total review pairs: 71
External pairs: 63
Internal pairs: 8
Unique components: 56

PAIR DISTRIBUTION BY PRIORITY


,priority,pair_type,pairs
0,P1_CALIBRATION,EXTERNAL,22
1,P1_CALIBRATION,INTERNAL,6
2,P2_PILOT,EXTERNAL,2
3,P3_TRAINING_INTERNAL,EXTERNAL,39
4,P3_TRAINING_INTERNAL,INTERNAL,2



FIRST 12 ADJUDICATION RECORDS


,pair_id,component_id,pair_type,record_a,record_b,partition_b,cosine,sequence,jaccard,priority,reviewer_1,reviewer_2,final_decision,review_notes
0,PAIR-001,C001,EXTERNAL,8972,9101,anchor,0.97701,0.96367,0.92500,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
1,PAIR-002,C001,EXTERNAL,8972,9584,training,0.90798,0.83871,0.71053,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
2,PAIR-003,C002,EXTERNAL,18762,18760,anchor,0.94556,0.94737,0.72414,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
3,PAIR-004,C003,EXTERNAL,33781,33762,anchor,0.91553,0.86471,0.83019,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
4,PAIR-005,C004,EXTERNAL,84216,83997,training,0.89243,0.86747,0.64865,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
5,PAIR-006,C004,EXTERNAL,84216,84524,anchor,0.89135,0.84156,0.61765,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
6,PAIR-007,C005,EXTERNAL,98008,97990,anchor,0.98932,0.95175,0.88235,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
7,PAIR-008,C006,EXTERNAL,103583,103660,anchor,0.89724,0.87649,0.80000,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
8,PAIR-009,C007,EXTERNAL,107847,107824,anchor,0.98194,0.95522,0.94444,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,
9,PAIR-010,C008,EXTERNAL,6540,4514,development,0.91100,0.89573,0.54902,P1_CALIBRATION,PENDING,PENDING,UNASSESSED,



Human decisions recorded: 0
Automatic duplicate removal: NO
Reference labels accessed: NO
Files modified: NO
Holdout frozen: NO


In [8]:

# Human Adjudication — Component C001
# Interactive review; decisions remain in memory only.

from IPython.display import clear_output
import re

VALID_INPUTS = {
    "D": "DUPLICATE",
    "R": "RELATED_DISTINCT",
    "F": "DIFFERENT",
    "U": "UNCERTAIN"
}

holdout_lookup = holdout_check.set_index(
    "record_id"
)["text"]

comparison_lookup = comparison.set_index(
    "record_id"
)["text"]

selected = adjudication_df[
    adjudication_df["component_id"] == "C001"
].sort_values("pair_id").copy()

assert set(selected["pair_id"]) == {
    "PAIR-001", "PAIR-002"
}

def readable(text):
    return re.sub(r"\s+", " ", str(text)).strip()

review_entries = []

for row in selected.itertuples(index=False):

    print("=" * 65)
    print("PAIR:", row.pair_id)
    print("Component:", row.component_id)
    print("Cosine similarity:", round(row.cosine, 5))
    print("=" * 65)

    print("\nHOLDOUT ARTICLE")
    print(readable(
        holdout_lookup.loc[row.record_a]
    ))

    print("\nCOMPARISON ARTICLE")
    print("Partition:", row.partition_b)
    print(readable(
        comparison_lookup.loc[row.record_b]
    ))

    print("\nD = Duplicate")
    print("R = Related but distinct")
    print("F = Different")
    print("U = Uncertain")

    while True:
        decision = input(
            "\nYour decision (D/R/F/U): "
        ).strip().upper()

        if decision in VALID_INPUTS:
            break

        print("Invalid decision. Try again.")

    while True:
        notes = input(
            "Brief evidence or reason: "
        ).strip()

        if notes:
            break

        print("Please enter a short explanation.")

    review_entries.append({
        "pair_id": row.pair_id,
        "decision": VALID_INPUTS[decision],
        "notes": notes
    })

# Clear full article text from saved cell output.
clear_output(wait=False)

for entry in review_entries:
    mask = (
        adjudication_df["pair_id"]
        == entry["pair_id"]
    )

    assert mask.sum() == 1

    adjudication_df.loc[
        mask, "reviewer_1"
    ] = entry["decision"]

    adjudication_df.loc[
        mask, "review_notes"
    ] = entry["notes"]

# Reviewer 2 and consensus remain pending.
assert adjudication_df.loc[
    adjudication_df["pair_id"].isin(
        {"PAIR-001", "PAIR-002"}
    ),
    "reviewer_2"
].eq("PENDING").all()

assert adjudication_df["final_decision"].eq(
    "UNASSESSED"
).all()

print("INITIAL HUMAN REVIEW RECORDED")
print("-----------------------------")
print("Component: C001")
print("Pairs reviewed:", len(review_entries))

display(
    adjudication_df.loc[
        adjudication_df["pair_id"].isin(
            {"PAIR-001", "PAIR-002"}
        ),
        [
            "pair_id",
            "component_id",
            "reviewer_1",
            "reviewer_2",
            "final_decision",
            "review_notes"
        ]
    ].reset_index(drop=True)
)

print("\nConsensus decisions: 0")
print("Reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


INITIAL HUMAN REVIEW RECORDED
-----------------------------
Component: C001
Pairs reviewed: 2


,pair_id,component_id,reviewer_1,reviewer_2,final_decision,review_notes
0,PAIR-001,C001,DUPLICATE,PENDING,UNASSESSED,Near-identical headline and article content; o...
1,PAIR-002,C001,DUPLICATE,PENDING,UNASSESSED,Same news event with substantially overlapping...



Consensus decisions: 0
Reference labels accessed: NO
Files modified: NO
Holdout frozen: NO


In [9]:

# Independent Human Review Packet Preparation
# Private files only; no reference labels or prior decisions.

from pathlib import Path
import pandas as pd

REVIEW_DIR = BASE / "outputs" / "private_human_review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

holdout_texts = holdout_check.set_index(
    "record_id"
)["text"]

comparison_texts = comparison.set_index(
    "record_id"
)["text"]

rows = []

for row in adjudication_df.itertuples(index=False):

    text_a = str(
        holdout_texts.loc[int(row.record_a)]
    )

    if row.partition_b == "holdout":
        text_b = str(
            holdout_texts.loc[int(row.record_b)]
        )
    else:
        text_b = str(
            comparison_texts.loc[int(row.record_b)]
        )

    rows.append({
        "pair_id": row.pair_id,
        "article_a": text_a,
        "article_b": text_b,
        "decision_D_R_F_U": "",
        "reason": ""
    })

review_packet = pd.DataFrame(rows)

assert len(review_packet) == 71
assert review_packet["pair_id"].is_unique
assert review_packet["article_a"].notna().all()
assert review_packet["article_b"].notna().all()

# Separate, randomly ordered files
reviewers = {
    "islam": 101,
    "yousuf": 202
}

for reviewer, seed in reviewers.items():

    packet = review_packet.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    file_path = REVIEW_DIR / f"{reviewer}_review.csv"

    packet.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    # Verify the saved file
    saved = pd.read_csv(file_path)

    assert len(saved) == 71
    assert set(saved["pair_id"]) == set(
        review_packet["pair_id"]
    )
    assert saved["decision_D_R_F_U"].isna().all()

    print(f"Created: {file_path.name} | 71 pairs")

print("\nINDEPENDENT REVIEW PACKETS CREATED")
print("Reviewers: Islam and Yousuf")
print("Similarity scores exported: NO")
print("Previous decisions exported: NO")
print("Reference labels accessed: NO")
print("Original dataset files modified: NO")
print("Holdout frozen: NO")
print("\nPrivate folder:", REVIEW_DIR)


Created: islam_review.csv | 71 pairs
Created: yousuf_review.csv | 71 pairs

INDEPENDENT REVIEW PACKETS CREATED
Reviewers: Islam and Yousuf
Similarity scores exported: NO
Previous decisions exported: NO
Reference labels accessed: NO
Original dataset files modified: NO
Holdout frozen: NO

Private folder: /content/drive/MyDrive/S_SLTE_Thesis/outputs/private_human_review


In [10]:

# Load Independent Human Review Files
# Source: Private Google Drive research directory

import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REVIEW_DIR = Path(
    "/content/drive/MyDrive/S_SLTE_Thesis/"
    "outputs/private_human_review"
)

islam_path = REVIEW_DIR / "islam_review_english.xlsx"
yousuf_path = REVIEW_DIR / "yousuf_review_english.xlsx"

assert islam_path.exists(), "Islam review file missing"
assert yousuf_path.exists(), "Yousuf review file missing"

islam_review = pd.read_excel(islam_path)
yousuf_review = pd.read_excel(yousuf_path)

required_columns = {
    "pair_id",
    "article_a",
    "article_b",
    "decision_D_R_F_U",
    "reason"
}

for name, df in [
    ("Islam", islam_review),
    ("Yousuf", yousuf_review)
]:
    assert len(df) == 71
    assert required_columns.issubset(df.columns)
    assert df["pair_id"].is_unique
    assert df["reason"].notna().all()

    decisions = (
        df["decision_D_R_F_U"]
        .astype(str).str.strip().str.upper()
    )

    assert decisions.isin({"D", "R", "F", "U"}).all()
    print(f"{name}: {len(df)} valid review records")

print("\nREVIEW FILES LOADED SUCCESSFULLY")
print("Source: Google Drive")
print("Reference labels accessed: NO")
print("Original files modified: NO")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Islam: 71 valid review records
Yousuf: 71 valid review records

REVIEW FILES LOADED SUCCESSFULLY
Source: Google Drive
Reference labels accessed: NO
Original files modified: NO


In [11]:

# Inter-Annotator Agreement Analysis
# Reuses review DataFrames loaded from Google Drive.
# No original files are modified.

import pandas as pd
from sklearn.metrics import cohen_kappa_score

VALID_LABELS = ["D", "R", "F", "U"]

def prepare_review(df, reviewer_name):
    data = df.copy()

    data["pair_id"] = (
        data["pair_id"].astype(str).str.strip()
    )

    data["decision_D_R_F_U"] = (
        data["decision_D_R_F_U"]
        .astype(str).str.strip().str.upper()
    )

    assert len(data) == 71
    assert data["pair_id"].is_unique
    assert data["decision_D_R_F_U"].isin(
        VALID_LABELS
    ).all()

    assert data["reason"].notna().all()
    assert data["reason"].astype(str).str.strip().ne(
        ""
    ).all()

    return data.set_index("pair_id").sort_index()

islam_data = prepare_review(
    islam_review, "Islam"
)

yousuf_data = prepare_review(
    yousuf_review, "Yousuf"
)

# Match by pair_id, not spreadsheet row position.
assert islam_data.index.equals(yousuf_data.index)

# Both reviewers must have assessed the same texts.
for column in ["article_a", "article_b"]:
    assert islam_data[column].equals(
        yousuf_data[column]
    ), f"Article mismatch: {column}"

agreement_df = pd.DataFrame({
    "islam_decision": islam_data[
        "decision_D_R_F_U"
    ],
    "yousuf_decision": yousuf_data[
        "decision_D_R_F_U"
    ]
})

agreement_df["agreed"] = (
    agreement_df["islam_decision"]
    == agreement_df["yousuf_decision"]
)

total = len(agreement_df)
matches = int(agreement_df["agreed"].sum())
disagreement_df = agreement_df[
    ~agreement_df["agreed"]
].copy()

observed_agreement = matches / total

kappa = cohen_kappa_score(
    agreement_df["islam_decision"],
    agreement_df["yousuf_decision"],
    labels=VALID_LABELS
)

confusion = pd.crosstab(
    agreement_df["islam_decision"],
    agreement_df["yousuf_decision"],
    rownames=["Islam"],
    colnames=["Yousuf"]
).reindex(
    index=VALID_LABELS,
    columns=VALID_LABELS,
    fill_value=0
)

assert confusion.to_numpy().sum() == 71

print("INTER-ANNOTATOR AGREEMENT ANALYSIS")
print("----------------------------------")
print("Total reviewed pairs:", total)
print("Matching decisions:", matches)
print("Disagreements:", len(disagreement_df))
print(f"Observed agreement: {observed_agreement:.2%}")
print(f"Descriptive Cohen's Kappa: {kappa:.4f}")

print("\nDECISION CONFUSION MATRIX")
display(confusion)

print("\nDISAGREEMENT CASES")
display(
    disagreement_df[
        ["islam_decision", "yousuf_decision"]
    ].reset_index()
)

print("\nVALIDATION STATUS")
print("Article alignment: PASSED")
print("Missing decisions: NONE")
print("Independent review provenance: NOT VERIFIED")
print("Final consensus: PENDING")
print("Reference labels accessed: NO")
print("Original files modified: NO")
print("Holdout frozen: NO")


INTER-ANNOTATOR AGREEMENT ANALYSIS
----------------------------------
Total reviewed pairs: 71
Matching decisions: 68
Disagreements: 3
Observed agreement: 95.77%
Descriptive Cohen's Kappa: 0.8913

DECISION CONFUSION MATRIX


Yousuf,D,R,F,U
Islam,,,,
D,52,0,0,0
R,2,0,0,0
F,0,1,16,0
U,0,0,0,0



DISAGREEMENT CASES


,pair_id,islam_decision,yousuf_decision
0,PAIR-032,R,D
1,PAIR-043,F,R
2,PAIR-057,R,D



VALIDATION STATUS
Article alignment: PASSED
Missing decisions: NONE
Independent review provenance: NOT VERIFIED
Final consensus: PENDING
Reference labels accessed: NO
Original files modified: NO
Holdout frozen: NO


In [12]:

# Blinded Disagreement Adjudication Packet
# No label access, no automatic decisions.
# Private Google Drive output.

from pathlib import Path
import pandas as pd

REVIEW_DIR = (
    BASE / "outputs" / "private_human_review"
)

DISAGREEMENT_IDS = [
    "PAIR-032",
    "PAIR-043",
    "PAIR-057"
]

assert set(disagreement_df.index) == set(
    DISAGREEMENT_IDS
)

assert islam_data.index.equals(yousuf_data.index)

# Build blind adjudication packet.
# Do not expose earlier reviewer decisions.
records = []

for pair_id in DISAGREEMENT_IDS:

    source = islam_data.loc[pair_id]

    # Verify both researchers reviewed identical text.
    for col in ["article_a", "article_b"]:
        assert (
            islam_data.loc[pair_id, col]
            == yousuf_data.loc[pair_id, col]
        )

    records.append({
        "pair_id": pair_id,
        "article_a": str(source["article_a"]),
        "article_b": str(source["article_b"]),
        "decision_D_R_F_U": "",
        "reason": ""
    })

adjudication_packet = pd.DataFrame(records)

assert len(adjudication_packet) == 3
assert adjudication_packet["pair_id"].is_unique

# Save to a separate private Excel file.
output_path = (
    REVIEW_DIR /
    "disagreement_adjudication_blind_v1.xlsx"
)

# Prevent accidental replacement.
assert not output_path.exists(), (
    "The file already exists. "
    "Use a new versioned filename if needed."
)

adjudication_packet.to_excel(
    output_path,
    index=False
)

# Verify saved contents.
saved = pd.read_excel(output_path)

assert len(saved) == 3
assert set(saved["pair_id"]) == set(
    DISAGREEMENT_IDS
)

assert saved["decision_D_R_F_U"].isna().all()

assert not {
    "islam_decision",
    "yousuf_decision",
    "cosine",
    "jaccard",
    "sequence"
}.intersection(saved.columns)

print("BLINDED ADJUDICATION PACKET CREATED")
print("-----------------------------------")
print("Total disagreement pairs:", len(saved))
print("Prior reviewer decisions included: NO")
print("Similarity scores included: NO")
print("Reference labels accessed: NO")
print("Original review files modified: NO")
print("Final consensus recorded: NO")
print("Holdout frozen: NO")
print("\nPrivate file:")
print(output_path)


BLINDED ADJUDICATION PACKET CREATED
-----------------------------------
Total disagreement pairs: 3
Prior reviewer decisions included: NO
Similarity scores included: NO
Reference labels accessed: NO
Original review files modified: NO
Final consensus recorded: NO
Holdout frozen: NO

Private file:
/content/drive/MyDrive/S_SLTE_Thesis/outputs/private_human_review/disagreement_adjudication_blind_v1.xlsx


In [13]:

# Discussion-Based Consensus Adjudication
# Prepares a private worksheet for the two researchers.
# No automatic consensus or dataset modification.

from pathlib import Path
import pandas as pd

REVIEW_DIR = (
    BASE / "outputs" / "private_human_review"
)

DISAGREEMENT_IDS = [
    "PAIR-032",
    "PAIR-043",
    "PAIR-057"
]

assert set(disagreement_df.index) == set(
    DISAGREEMENT_IDS
)

records = []

for pair_id in DISAGREEMENT_IDS:

    islam_row = islam_data.loc[pair_id]
    yousuf_row = yousuf_data.loc[pair_id]

    # The reviewers must have assessed identical articles.
    assert islam_row["article_a"] == yousuf_row["article_a"]
    assert islam_row["article_b"] == yousuf_row["article_b"]

    records.append({
        "pair_id": pair_id,
        "article_a": islam_row["article_a"],
        "article_b": islam_row["article_b"],
        "islam_decision": islam_row["decision_D_R_F_U"],
        "islam_reason": islam_row["reason"],
        "yousuf_decision": yousuf_row["decision_D_R_F_U"],
        "yousuf_reason": yousuf_row["reason"],
        "final_decision_D_R_F_U": "",
        "consensus_reason": "",
        "resolution_status": "PENDING_DISCUSSION"
    })

consensus_sheet = pd.DataFrame(records)

assert len(consensus_sheet) == 3
assert consensus_sheet["pair_id"].is_unique
assert consensus_sheet[
    "final_decision_D_R_F_U"
].eq("").all()

output_path = (
    REVIEW_DIR /
    "consensus_adjudication_v1.xlsx"
)

# Preserve an existing worksheet rather than overwrite it.
assert not output_path.exists(), (
    "Consensus worksheet already exists. "
    "Use the existing file or a new versioned filename."
)

consensus_sheet.to_excel(
    output_path,
    index=False
)

# Verify the saved worksheet.
saved = pd.read_excel(output_path)

assert len(saved) == 3
assert saved["final_decision_D_R_F_U"].isna().all()

print("CONSENSUS ADJUDICATION WORKSHEET CREATED")
print("----------------------------------------")
print("Researchers: Islam and Yousuf")
print("Disagreement pairs:", len(saved))
print("Automatic decisions:", 0)
print("Consensus decisions:", 0)
print("Resolution method: TWO_REVIEWER_DISCUSSION")

print("\nDISAGREEMENTS TO RESOLVE")
display(
    saved[
        [
            "pair_id",
            "islam_decision",
            "yousuf_decision",
            "resolution_status"
        ]
    ]
)

print("\nPrivate worksheet:", output_path)
print("Original review files modified: NO")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")


CONSENSUS ADJUDICATION WORKSHEET CREATED
----------------------------------------
Researchers: Islam and Yousuf
Disagreement pairs: 3
Automatic decisions: 0
Consensus decisions: 0
Resolution method: TWO_REVIEWER_DISCUSSION

DISAGREEMENTS TO RESOLVE


,pair_id,islam_decision,yousuf_decision,resolution_status
0,PAIR-032,R,D,PENDING_DISCUSSION
1,PAIR-043,F,R,PENDING_DISCUSSION
2,PAIR-057,R,D,PENDING_DISCUSSION



Private worksheet: /content/drive/MyDrive/S_SLTE_Thesis/outputs/private_human_review/consensus_adjudication_v1.xlsx
Original review files modified: NO
Reference labels accessed: NO
Holdout frozen: NO


In [14]:

# Final Consensus Validation
# No dataset modifications or file writes.

import pandas as pd

consensus_path = (
    REVIEW_DIR / "consensus_adjudication_v1.xlsx"
)

consensus = pd.read_excel(consensus_path)

assert len(consensus) == 3
assert consensus["pair_id"].is_unique
assert set(consensus["pair_id"]) == set(
    disagreement_df.index
)

assert consensus["resolution_status"].eq(
    "RESOLVED"
).all()

assert consensus["final_decision_D_R_F_U"].isin(
    ["D", "R", "F", "U"]
).all()

assert consensus["consensus_reason"].notna().all()
assert consensus["consensus_reason"].astype(
    str
).str.strip().ne("").all()

# Start with both reviewers' original decisions
final_review = agreement_df.copy()

final_review["final_decision"] = (
    final_review["islam_decision"]
)

final_review["resolution_method"] = (
    "REVIEWER_AGREEMENT"
)

# Update only resolved disagreements
for row in consensus.itertuples(index=False):

    pair_id = row.pair_id

    assert not final_review.loc[
        pair_id, "agreed"
    ]

    final_review.loc[
        pair_id, "final_decision"
    ] = row.final_decision_D_R_F_U

    final_review.loc[
        pair_id, "resolution_method"
    ] = "DISCUSSION_CONSENSUS"

assert len(final_review) == 71
assert final_review["final_decision"].notna().all()

print("FINAL CONSENSUS VALIDATION PASSED")
print("---------------------------------")
print("Total reviewed pairs:", len(final_review))
print("Reviewer agreements:", int(final_review["agreed"].sum()))
print("Discussion-resolved disagreements:", 3)
print("Unresolved disagreements:", 0)

print("\nFINAL DECISION DISTRIBUTION")
display(
    final_review["final_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="pairs")
)

print("\nCONSENSUS DECISIONS")
display(
    final_review.loc[
        ["PAIR-032", "PAIR-043", "PAIR-057"],
        ["islam_decision", "yousuf_decision",
         "final_decision", "resolution_method"]
    ]
)

print("\nSource files modified: NO")
print("Reference labels accessed: NO")
print("Holdout frozen: NO")
print("Reviewer independence: NOT VERIFIED")


FINAL CONSENSUS VALIDATION PASSED
---------------------------------
Total reviewed pairs: 71
Reviewer agreements: 68
Discussion-resolved disagreements: 3
Unresolved disagreements: 0

FINAL DECISION DISTRIBUTION


,decision,pairs
0,D,53
1,F,17
2,R,1



CONSENSUS DECISIONS


,islam_decision,yousuf_decision,final_decision,resolution_method
pair_id,,,,
PAIR-032,R,D,R,DISCUSSION_CONSENSUS
PAIR-043,F,R,F,DISCUSSION_CONSENSUS
PAIR-057,R,D,D,DISCUSSION_CONSENSUS



Source files modified: NO
Reference labels accessed: NO
Holdout frozen: NO
Reviewer independence: NOT VERIFIED


In [16]:

# Duplicate-Aware Holdout Policy Analysis
# Corrected: prevents column-name collisions.
# Read-only: no dataset changes or file writes.

import pandas as pd

assert len(adjudication_df) == 71
assert len(final_review) == 71

# Remove only old working decision columns from
# a copy of the original adjudication metadata.
metadata = adjudication_df.drop(
    columns=[
        "final_decision",
        "resolution_method",
        "policy_flag"
    ],
    errors="ignore"
).copy()

# Merge verified consensus decisions.
policy_audit = metadata.merge(
    final_review[
        ["final_decision", "resolution_method"]
    ],
    left_on="pair_id",
    right_index=True,
    how="left",
    validate="one_to_one"
)

assert len(policy_audit) == 71
assert policy_audit["final_decision"].notna().all()

assert policy_audit["final_decision"].value_counts().to_dict() == {
    "D": 53,
    "F": 17,
    "R": 1
}

# Normalize metadata for robust classification.
policy_audit["pair_type"] = (
    policy_audit["pair_type"]
    .astype(str).str.strip().str.upper()
)

policy_audit["partition_b_normalized"] = (
    policy_audit["partition_b"]
    .fillna("")
    .astype(str).str.strip().str.lower()
)

def classify_policy(row):
    decision = row["final_decision"]

    if decision == "F":
        return "NO_DUPLICATE_ACTION"

    if decision == "R":
        return "RELATED_DISTINCT_REVIEW"

    if decision != "D":
        return "UNRESOLVED_DECISION"

    if row["pair_type"] == "INTERNAL":
        return "INTERNAL_HOLDOUT_RESOLUTION"

    partition = row["partition_b_normalized"]

    if partition in {"anchor", "development", "pilot"}:
        return "HOLDOUT_REPLACEMENT_CANDIDATE"

    if partition in {"training", "train"}:
        return "TRAINING_EXCLUSION_CANDIDATE"

    return "UNKNOWN_PARTITION_REQUIRES_REVIEW"


policy_audit["policy_flag"] = policy_audit.apply(
    classify_policy, axis=1
)

# Block unrecognized metadata rather than
# silently assigning an incorrect policy.
unknown = policy_audit[
    policy_audit["policy_flag"].isin([
        "UNRESOLVED_DECISION",
        "UNKNOWN_PARTITION_REQUIRES_REVIEW"
    ])
]

assert unknown.empty, (
    "Unrecognized metadata:\n"
    + unknown[
        ["pair_id", "pair_type", "partition_b"]
    ].to_string(index=False)
)

protected_duplicate_pairs = policy_audit[
    policy_audit["policy_flag"]
    == "HOLDOUT_REPLACEMENT_CANDIDATE"
]

training_duplicate_pairs = policy_audit[
    policy_audit["policy_flag"]
    == "TRAINING_EXCLUSION_CANDIDATE"
]

internal_duplicate_pairs = policy_audit[
    policy_audit["policy_flag"]
    == "INTERNAL_HOLDOUT_RESOLUTION"
]

holdout_replacement_candidates = set(
    protected_duplicate_pairs["record_a"].astype(int)
)

training_exclusion_candidates = set(
    training_duplicate_pairs["record_b"].astype(int)
)

internal_review_ids = (
    set(internal_duplicate_pairs["record_a"].astype(int))
    | set(internal_duplicate_pairs["record_b"].astype(int))
)

# Validate against the in-memory proposed splits.
known_holdout_ids = set(
    holdout_df["record_id"].astype(int)
)

known_training_ids = set(
    clean_training_df["record_id"].astype(int)
)

assert holdout_replacement_candidates.issubset(
    known_holdout_ids
)

assert internal_review_ids.issubset(
    known_holdout_ids
)

assert training_exclusion_candidates.issubset(
    known_training_ids
)

print("HOLDOUT POLICY CANDIDATE ANALYSIS PASSED")
print("-----------------------------------------")
print("Total reviewed pairs:", len(policy_audit))
print("Duplicate pairs: 53")
print("Different pairs: 17")
print("Related-distinct pairs: 1")

print("\nDECISIONS BY PAIR TYPE")
display(pd.crosstab(
    policy_audit["pair_type"],
    policy_audit["final_decision"]
))

print("\nPROPOSED POLICY FLAGS")
display(
    policy_audit["policy_flag"]
    .value_counts()
    .rename_axis("policy_flag")
    .reset_index(name="pairs")
)

print("\nUNIQUE RECORD CANDIDATES")
print(
    "Holdout replacement candidates:",
    len(holdout_replacement_candidates)
)
print(
    "Training exclusion candidates:",
    len(training_exclusion_candidates)
)
print(
    "Holdout records needing internal resolution:",
    len(internal_review_ids)
)

print("\nOriginal datasets modified: NO")
print("Reference labels accessed: NO")
print("Records removed: 0")
print("Holdout frozen: NO")
print("Policy status: DRAFT_REQUIRES_REVIEW")


HOLDOUT POLICY CANDIDATE ANALYSIS PASSED
-----------------------------------------
Total reviewed pairs: 71
Duplicate pairs: 53
Different pairs: 17
Related-distinct pairs: 1

DECISIONS BY PAIR TYPE


final_decision,D,F,R
pair_type,,,
EXTERNAL,51,11,1
INTERNAL,2,6,0



PROPOSED POLICY FLAGS


,policy_flag,pairs
0,TRAINING_EXCLUSION_CANDIDATE,37
1,NO_DUPLICATE_ACTION,17
2,HOLDOUT_REPLACEMENT_CANDIDATE,14
3,INTERNAL_HOLDOUT_RESOLUTION,2
4,RELATED_DISTINCT_REVIEW,1



UNIQUE RECORD CANDIDATES
Holdout replacement candidates: 14
Training exclusion candidates: 37
Holdout records needing internal resolution: 4

Original datasets modified: NO
Reference labels accessed: NO
Records removed: 0
Holdout frozen: NO
Policy status: DRAFT_REQUIRES_REVIEW
